In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import numpy as np
import pandas as pd
# from tqdm.auto import tqdm # 暂时不用 tqdm

# --- 配置参数 ---
MODEL_PATH = "./absa_baseline_model/checkpoint-150"
INPUT_FILE = 'combined_comment.csv' # 确保这是你的文件名
OUTPUT_FILE_PREDICTIONS = 'comment_predictions_pipeline_progress.csv' # 改个名，区分一下
PIPELINE_BATCH_SIZE = 16 # CPU 上可以设小点，比如 8 或 16
PROGRESS_INTERVAL = 1000 # 每处理 1000 条吱一声

# --- 1. 加载模型和分词器 (基本不变) ---
print(f"Loading model and tokenizer from {MODEL_PATH}...")
try:
    device_id = -1 # 强制 CPU
    device_name = "cpu"
    print(f"Using device: {device_name}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)

    print("Using manually defined label list...")
    label_list = ["O",
                  "B-ASP-POS", "I-ASP-POS", "E-ASP-POS", "S-ASP-POS",
                  "B-ASP-NEG", "I-ASP-NEG", "E-ASP-NEG", "S-ASP-NEG",
                  "B-ASP-NEU", "I-ASP-NEU", "E-ASP-NEU", "S-ASP-NEU"]
    id2label = {i: label for i, label in enumerate(label_list)}
    model.config.id2label = id2label
    model.config.label2id = {label: i for i, label in enumerate(label_list)}

    print(f"Labels: {label_list}")

except Exception as e:
    print(f"Error loading model or tokenizer: {e}")
    exit()

# --- 2. 创建 Pipeline (基本不变) ---
print("Creating token classification pipeline...")
token_classifier = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=device_id, # 确保传入 -1
    batch_size=PIPELINE_BATCH_SIZE,
    aggregation_strategy="none"
)

# --- 3. 定义后处理函数 (不变) ---
def post_process_pipeline_output(predictions):
    # (这里的代码和之前方案一完全一样，我就省略了，请确保你那边有这段代码)
    results = []
    current_aspect = ""
    current_sentiment = ""
    last_end = -1

    for entity in predictions:
        label = entity['entity']
        word = entity['word']
        start = entity['start']
        end = entity['end']

        if word.startswith("##"): word = word[2:]
        word = word.replace(' ', '')
        if not word: continue

        tag = label[0]
        sentiment = label.split("-")[-1] if tag != 'O' else None

        if tag == 'B' or tag == 'S':
            if current_aspect: results.append({"aspect": current_aspect, "sentiment": current_sentiment})
            current_aspect = word
            current_sentiment = sentiment
            last_end = end
        elif (tag == 'I' or tag == 'E') and current_aspect and sentiment == current_sentiment:
             current_aspect += word
             last_end = end
        else:
            if current_aspect: results.append({"aspect": current_aspect, "sentiment": current_sentiment})
            current_aspect = ""; current_sentiment = ""; last_end = -1

        if tag == 'S' or tag == 'E':
             if current_aspect: results.append({"aspect": current_aspect, "sentiment": current_sentiment})
             current_aspect = ""; current_sentiment = ""; last_end = -1

    if current_aspect: results.append({"aspect": current_aspect, "sentiment": current_sentiment})

    final_results = []
    seen_aspects = set()
    for res in results:
        key = (res['aspect'], res['sentiment'])
        if key not in seen_aspects:
            final_results.append(res)
            seen_aspects.add(key)
    return final_results


# --- 4. 处理整个文件 (修改版：分块处理) ---
print(f"\n--- Predicting on file using Pipeline (chunked): {INPUT_FILE} ---")

# 引入 tqdm (这回它能正常工作了)
from tqdm.auto import tqdm

# 定义你希望每次处理的块大小
CHUNKSIZE = 1000 # 每次加载和处理 1000 条
total_processed_count = 0
is_first_chunk = True # 用于标记是否是第一次写入文件（需要写表头）

try:
    # 使用 pd.read_csv 的 chunksize 参数，它会返回一个迭代器
    chunk_iterator = pd.read_csv(INPUT_FILE, chunksize=CHUNKSIZE)
    
    print(f"Starting chunked processing (chunk size = {CHUNKSIZE})...")

    # 循环遍历每个数据块
    for chunk_df in chunk_iterator:
        if '客户反馈' not in chunk_df.columns:
            raise ValueError("Column '客户反馈' not found in the input CSV.")

        # 1. 获取当前块的评论
        comments = chunk_df['客户反馈'].fillna("").tolist()
        current_chunk_size = len(comments)
        
        print(f"  Processing chunk: {total_processed_count + 1} to {total_processed_count + current_chunk_size} comments...")

        # 2. 用 Pipeline 处理这 1000 条 (这会很快)
        pipeline_outputs = []
        # 在这里用 tqdm，可以实时看到这 1000 条的进度
        for output in tqdm(token_classifier(comments), total=len(comments), desc="  Chunk progress"):
            pipeline_outputs.append(output)

        # 3. 后处理
        chunk_df['predictions'] = [post_process_pipeline_output(output) for output in pipeline_outputs]

        # 4. 增量写入文件
        if is_first_chunk:
            # 第一个块：创建文件并写入表头
            chunk_df.to_csv(OUTPUT_FILE_PREDICTIONS, index=False, encoding='utf-8-sig', mode='w')
            is_first_chunk = False
        else:
            # 后续块：追加内容，不写表头
            chunk_df.to_csv(OUTPUT_FILE_PREDICTIONS, index=False, encoding='utf-8-sig', mode='a', header=False)

        total_processed_count += current_chunk_size
        print(f"  Finished chunk. Total processed so far: {total_processed_count} comments.")

    print("\n--- All chunks processed! ---")
    print(f"Predictions saved to {OUTPUT_FILE_PREDICTIONS}")

except FileNotFoundError:
    print(f"Error: Input file not found at {INPUT_FILE}")
except Exception as e:
    print(f"An error occurred while processing the file: {e}")

Device set to use cpu


Loading model and tokenizer from ./absa_baseline_model/checkpoint-150...
Using device: cpu
Using manually defined label list...
Labels: ['O', 'B-ASP-POS', 'I-ASP-POS', 'E-ASP-POS', 'S-ASP-POS', 'B-ASP-NEG', 'I-ASP-NEG', 'E-ASP-NEG', 'S-ASP-NEG', 'B-ASP-NEU', 'I-ASP-NEU', 'E-ASP-NEU', 'S-ASP-NEU']
Creating token classification pipeline...

--- Predicting on file using Pipeline (chunked): combined_comment.csv ---
Starting chunked processing (chunk size = 1000)...
  Processing chunk: 1 to 1000 comments...


  Chunk progress:   0%|          | 0/1000 [00:00<?, ?it/s]

  Finished chunk. Total processed so far: 1000 comments.
  Processing chunk: 1001 to 2000 comments...


  Chunk progress:   0%|          | 0/1000 [00:00<?, ?it/s]

...

  Finished chunk. Total processed so far: 200000 comments.
  Processing chunk: 200001 to 201000 comments...


  Chunk progress:   0%|          | 0/1000 [00:00<?, ?it/s]

  Finished chunk. Total processed so far: 201000 comments.
  Processing chunk: 201001 to 202000 comments...


  Chunk progress:   0%|          | 0/1000 [00:00<?, ?it/s]

  Finished chunk. Total processed so far: 202000 comments.
  Processing chunk: 202001 to 203000 comments...
